# VisClick — Phase 4.2 / D-01 (step 2 of 2): fine-tune DETR-R50 on hand-corrected desktop

**Prerequisites:**
- `09_detr_source.ipynb` ran to completion: `<DRIVE>/weights/baseline_source_detr/best_source_detr_r50.pt` exists.
- The hand-corrected zip is committed to the repo at `datasets/handcorrected_desktop_test/visclick3.yolov8.zip` (same file `scripts/run_cpv.py` uses).

**This notebook** (Phase 4.2 — D-01 — finishes the DETR transfer-learning lane):
1. Mount Drive → `git pull` → install deps.
2. **Unzip** the 8-image hand-corrected set into `/content/hc/` and convert YOLO labels → COCO JSON (one JSON; we use all 8 images for fine-tuning, given the budget is tiny we don't hold any out).
3. **Load** the source-best DETR checkpoint and **fine-tune** for 30 epochs at a 10× lower learning rate. Head-only fine-tune (freeze backbone) to match the YOLOv8 M3 recipe in `06_finetune_desktop.ipynb`.
4. **Save stable name** `desktop_finetune_detr/best_desktop_detr_r50.pt`.
5. **Evaluate** mAP@0.5 + mAP@0.5:0.95 on the same 8 images (training-set evaluation — sanity check that fine-tuning improved over zero-shot, **not** a generalisation claim).
6. **Compute CPV** with the same `scripts/run_cpv.py` protocol: predicted-box centre inside any GT box, counted per class and overall. Output `<DRIVE>/reports/tables/cpv_summary_detr.csv` and `cpv_per_image_detr.csv`.
7. Append a new row to `<DRIVE>/reports/tables/transfer_experiments.csv` so T-01 picks up the DETR fine-tune cell.

**Why fine-tune on all 8?** With n=8, holding a test split out leaves 6 train + 2 test which is too small for either to be meaningful. The dissertation's headline external validity for the detector comes from ScreenSpot (D-07, n=334), not the hand-corrected set. The hand-corrected mAP / CPV is reported as a *per-element recall* sanity check, same as the YOLOv8 row.

**Report.** Every step prints `REPORT ...` lines for the data form §4 row *Desktop fine-tune (DETR-R50)*.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
REPO = "https://github.com/HiranMadhu/visclick.git"
ROOT = "/content/visclick"
if not os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "clone", REPO, ROOT], check=True)
    print("Cloned to", ROOT)
else:
    subprocess.run(["git", "-C", ROOT, "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", ROOT, "pull", "--rebase", "origin", "main"], check=False)
    print("Pulled latest in", ROOT)
print("REPORT git_head =", subprocess.check_output(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
import sys, subprocess
# Pin Pillow below 12.0.0 (broken release; see 09_detr_source.ipynb cell 3 note).
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "Pillow==11.3.0",
     "transformers", "timm", "accelerate", "pycocotools", "torchmetrics"],
    check=False,
)
import torch, transformers, PIL
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("transformers:", transformers.__version__, "| Pillow:", PIL.__version__)
print("REPORT env | torch =", torch.__version__,
      "| cuda =", torch.cuda.is_available(),
      "| transformers =", transformers.__version__,
      "| pillow =", PIL.__version__)

## 10.1 — Unzip hand-corrected + convert to COCO

Unzips `datasets/handcorrected_desktop_test/visclick3.yolov8.zip` from the cloned repo into `/content/hc/`, then walks the `train/{images,labels}` split (8 images, 356 GT boxes total) and writes one COCO JSON.

This is the exact same data `scripts/run_cpv.py` uses, so the DETR CPV numbers are directly comparable to the YOLOv8 numbers in `reports/tables/cpv_summary.csv`.

In [ ]:
import os, zipfile, json, time
from PIL import Image

DRIVE   = "/content/drive/MyDrive/visclick"
CLASSES = ["button", "text", "text_input", "icon", "menu", "checkbox"]
CATEGORIES = [{"id": i, "name": n, "supercategory": "ui"} for i, n in enumerate(CLASSES)]

HC_ZIP  = "/content/visclick/datasets/handcorrected_desktop_test/visclick3.yolov8.zip"
HC_ROOT = "/content/hc"
HC_IMG  = os.path.join(HC_ROOT, "train", "images")
HC_LBL  = os.path.join(HC_ROOT, "train", "labels")
HC_JSON = os.path.join(HC_ROOT, "train.json")

if not os.path.isdir(HC_IMG):
    assert os.path.isfile(HC_ZIP), f"Missing zip {HC_ZIP}; pull repo or copy file."
    os.makedirs(HC_ROOT, exist_ok=True)
    with zipfile.ZipFile(HC_ZIP, "r") as zf:
        zf.extractall(HC_ROOT)
    print("unzipped hand-corrected ->", HC_ROOT)
else:
    print("hand-corrected already unzipped at", HC_ROOT)

images, annotations = [], []
img_id = ann_id = 0
for fn in sorted(os.listdir(HC_IMG)):
    if not fn.lower().endswith((".jpg", ".jpeg", ".png")):
        continue
    p = os.path.join(HC_IMG, fn)
    with Image.open(p) as im:
        w, h = im.size
    images.append({"id": img_id, "file_name": fn, "width": w, "height": h})
    stem = os.path.splitext(fn)[0]
    lbl_p = os.path.join(HC_LBL, stem + ".txt")
    if os.path.isfile(lbl_p):
        with open(lbl_p) as fh:
            for line in fh:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls = int(parts[0])
                cx, cy, bw, bh = (float(p) for p in parts[1:])
                x = max(0.0, (cx - bw / 2.0) * w)
                y = max(0.0, (cy - bh / 2.0) * h)
                bw_abs = min(w - x, bw * w)
                bh_abs = min(h - y, bh * h)
                if bw_abs <= 1 or bh_abs <= 1:
                    continue
                annotations.append({
                    "id": ann_id, "image_id": img_id, "category_id": cls,
                    "bbox": [x, y, bw_abs, bh_abs], "area": bw_abs * bh_abs, "iscrowd": 0,
                })
                ann_id += 1
    img_id += 1

with open(HC_JSON, "w") as fh:
    json.dump({"info": {"description": "VisClick hand-corrected (YOLO→COCO)"},
               "images": images, "annotations": annotations, "categories": CATEGORIES}, fh)

print(f"REPORT hc | images = {len(images)} | annotations = {len(annotations)} | json = {HC_JSON}")

## 10.2 — Build dataset + processor + load source-best weights

Reuses the same `VisClickCocoDetection` wrapper pattern from `09_detr_source.ipynb`. Loads the DETR-R50 architecture, then drops the source-best state dict on top.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import CocoDetection
from transformers import DetrImageProcessor, DetrForObjectDetection

HF_MODEL = "facebook/detr-resnet-50"
processor = DetrImageProcessor.from_pretrained(HF_MODEL)

class VisClickCocoDetection(CocoDetection):
    def __init__(self, img_folder, ann_file, processor):
        super().__init__(img_folder, ann_file)
        self.processor = processor

    def __getitem__(self, idx):
        img, target = super().__getitem__(idx)
        image_id = self.ids[idx]
        target = {"image_id": image_id, "annotations": target}
        encoding = self.processor(images=img, annotations=target, return_tensors="pt")
        return encoding["pixel_values"].squeeze(0), encoding["labels"][0]

def collate_fn(batch):
    pixel_values = [b[0] for b in batch]
    encoding = processor.pad(pixel_values, return_tensors="pt")
    labels = [b[1] for b in batch]
    return {"pixel_values": encoding["pixel_values"],
            "pixel_mask": encoding["pixel_mask"], "labels": labels}

hc_ds = VisClickCocoDetection(HC_IMG, HC_JSON, processor)
print(f"REPORT dataset | hand_corrected = {len(hc_ds)}")

# Load source-best.
SRC_BEST = os.path.join(DRIVE, "weights", "baseline_source_detr", "best_source_detr_r50.pt")
assert os.path.isfile(SRC_BEST), f"Missing {SRC_BEST}; run 09_detr_source.ipynb first."

device = "cuda" if torch.cuda.is_available() else "cpu"
model = DetrForObjectDetection.from_pretrained(
    HF_MODEL,
    num_labels=len(CLASSES),
    ignore_mismatched_sizes=True,
    id2label={i: n for i, n in enumerate(CLASSES)},
    label2id={n: i for i, n in enumerate(CLASSES)},
)
state = torch.load(SRC_BEST, map_location="cpu")
missing, unexpected = model.load_state_dict(state["model"], strict=False)
print("REPORT load_source_best | missing =", len(missing), "| unexpected =", len(unexpected))
model = model.to(device)

## 10.3 — Head-only fine-tune for 30 epochs

Matches `06_finetune_desktop.ipynb`'s recipe: freeze the backbone (`requires_grad = False` on all backbone params), train only the transformer encoder/decoder and class/box heads. LR is 10× lower than the source run because the head starts from the source-best, not random init.

In [ ]:
import os, json, time, shutil
from torch.optim import AdamW

PROJECT = os.path.join(DRIVE, "weights", "desktop_finetune_detr")
RUN_DIR = os.path.join(PROJECT, "run1")
WTS_DIR = os.path.join(RUN_DIR, "weights")
LAST_PT = os.path.join(WTS_DIR, "last.pt")
BEST_PT = os.path.join(WTS_DIR, "best.pt")
META_JS = os.path.join(WTS_DIR, "meta.json")
STABLE  = os.path.join(PROJECT, "best_desktop_detr_r50.pt")
os.makedirs(WTS_DIR, exist_ok=True)

FORCE_FRESH = False
EPOCHS      = 30
MICRO_BATCH = 2
ACCUM       = 4       # effective batch 8 (hand-corrected is only 8 images)
LR_HEAD     = 1e-5
LR_BACKBONE = 0.0     # frozen
WEIGHT_DECAY= 1e-4

# Freeze backbone parameters.
n_frozen = 0
for n, p in model.named_parameters():
    if "backbone" in n:
        p.requires_grad = False
        n_frozen += 1
print(f"REPORT freeze | n_backbone_params_frozen = {n_frozen}")

head_params = [p for n, p in model.named_parameters() if p.requires_grad]
optim = AdamW(head_params, lr=LR_HEAD, weight_decay=WEIGHT_DECAY)

train_loader = DataLoader(
    hc_ds, batch_size=MICRO_BATCH, shuffle=True,
    collate_fn=collate_fn, num_workers=2, pin_memory=True,
)

def _save_ckpt(model, path, meta):
    torch.save({"model": model.state_dict(), "meta": meta}, path)
    with open(META_JS, "w") as fh:
        json.dump(meta, fh)

start_epoch = 0
best_loss = float("inf")
if os.path.isfile(LAST_PT) and not FORCE_FRESH:
    print("last.pt found -> resuming:", LAST_PT)
    state = torch.load(LAST_PT, map_location="cpu")
    model.load_state_dict(state["model"], strict=False)
    if os.path.isfile(META_JS):
        try:
            with open(META_JS) as fh:
                m = json.load(fh)
                start_epoch = int(m.get("last_epoch", 0)) + 1
                best_loss   = float(m.get("best_loss", float("inf")))
        except (json.JSONDecodeError, OSError):
            pass
    model = model.to(device)

t0 = time.time()
for epoch in range(start_epoch, EPOCHS):
    model.train()
    ep_loss, ep_steps = 0.0, 0
    optim.zero_grad()
    for step, batch in enumerate(train_loader):
        pixel_values = batch["pixel_values"].to(device)
        pixel_mask = batch["pixel_mask"].to(device)
        labels = [{k: v.to(device) for k, v in t.items()} for t in batch["labels"]]
        out = model(pixel_values=pixel_values, pixel_mask=pixel_mask, labels=labels)
        loss = out.loss / ACCUM
        loss.backward()
        ep_loss += float(out.loss.detach())
        ep_steps += 1
        if (step + 1) % ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(head_params, 0.1)
            optim.step()
            optim.zero_grad()
    torch.nn.utils.clip_grad_norm_(head_params, 0.1)
    optim.step()
    optim.zero_grad()
    train_loss = ep_loss / max(1, ep_steps)
    meta = {"last_epoch": epoch, "best_loss": best_loss}
    _save_ckpt(model, LAST_PT, meta)
    if train_loss < best_loss:
        best_loss = train_loss
        meta["best_loss"] = best_loss
        _save_ckpt(model, BEST_PT, meta)
        improved = True
    else:
        improved = False
    print(f"REPORT epoch | n = {epoch+1:>2d}/{EPOCHS} "
          f"| train_loss = {train_loss:0.4f} | best = {best_loss:0.4f} | improved = {improved}")

elapsed = time.time() - t0
if os.path.isfile(BEST_PT):
    shutil.copy2(BEST_PT, STABLE)
    print("REPORT step = SAVE_STABLE | dst =", STABLE, "| bytes =", os.path.getsize(STABLE))
print(f"REPORT step = TRAIN | elapsed_s = {elapsed:0.0f} | run_dir = {RUN_DIR}")

## 10.4 — Evaluate: mAP@0.5 + mAP@0.5:0.95 + CPV

Two metrics:
- **mAP** via `torchmetrics.detection.MeanAveragePrecision`, same as 9.6.
- **CPV** (Central Point Validation) using the same protocol as `scripts/run_cpv.py`: for each GT box, mark hit=1 if **any** predicted box's centre lands inside it. Aggregated per class and overall. This is the *per-element recall* protocol that produced the 1.4% YOLOv8 number in `cpv_summary.csv`.

Conf threshold for CPV: 0.25 (matches the YOLOv8 default in `run_cpv.py`).

In [ ]:
import csv
from torchmetrics.detection import MeanAveragePrecision

# Reload best for eval.
if os.path.isfile(STABLE):
    state = torch.load(STABLE, map_location="cpu")
    model.load_state_dict(state["model"], strict=False)
elif os.path.isfile(BEST_PT):
    state = torch.load(BEST_PT, map_location="cpu")
    model.load_state_dict(state["model"], strict=False)
model = model.to(device).eval()

eval_loader = DataLoader(
    hc_ds, batch_size=MICRO_BATCH, shuffle=False,
    collate_fn=collate_fn, num_workers=2, pin_memory=True,
)

CONF_CPV = 0.25
metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
per_class_hit  = {c: 0 for c in CLASSES}
per_class_gt   = {c: 0 for c in CLASSES}
per_image_rows = []
overall_hit = overall_gt = overall_preds = 0

with torch.no_grad():
    for batch_idx, batch in enumerate(eval_loader):
        pixel_values = batch["pixel_values"].to(device)
        pixel_mask = batch["pixel_mask"].to(device)
        outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask)
        target_sizes = torch.stack([
            torch.tensor([int(t["orig_size"][0]), int(t["orig_size"][1])])
            for t in batch["labels"]
        ]).to(device)
        # For mAP, use a permissive threshold (0.05).
        results_map = processor.post_process_object_detection(
            outputs, threshold=0.05, target_sizes=target_sizes,
        )
        # For CPV, use the same conf=0.25 as run_cpv.py.
        results_cpv = processor.post_process_object_detection(
            outputs, threshold=CONF_CPV, target_sizes=target_sizes,
        )
        # mAP update.
        preds_map = [
            {"boxes": r["boxes"].cpu(), "scores": r["scores"].cpu(),
             "labels": r["labels"].cpu()}
            for r in results_map
        ]
        gts_map = []
        for t in batch["labels"]:
            h, w = int(t["orig_size"][0]), int(t["orig_size"][1])
            cxcywh = t["boxes"]
            x1 = (cxcywh[:, 0] - cxcywh[:, 2] / 2.0) * w
            y1 = (cxcywh[:, 1] - cxcywh[:, 3] / 2.0) * h
            x2 = (cxcywh[:, 0] + cxcywh[:, 2] / 2.0) * w
            y2 = (cxcywh[:, 1] + cxcywh[:, 3] / 2.0) * h
            gts_map.append({"boxes": torch.stack([x1, y1, x2, y2], dim=-1).cpu(),
                            "labels": t["class_labels"].cpu()})
        metric.update(preds_map, gts_map)
        # CPV update.
        for r_cpv, t, gt_map in zip(results_cpv, batch["labels"], gts_map):
            boxes = r_cpv["boxes"].cpu()
            centres = [((float(b[0]) + float(b[2])) / 2.0,
                         (float(b[1]) + float(b[3])) / 2.0)
                        for b in boxes]
            img_hit = img_gt = 0
            gt_boxes = gt_map["boxes"]
            gt_labels = gt_map["labels"]
            for j in range(gt_boxes.shape[0]):
                x1, y1, x2, y2 = (float(v) for v in gt_boxes[j].tolist())
                cls_idx = int(gt_labels[j])
                cls_name = CLASSES[cls_idx]
                per_class_gt[cls_name] += 1
                img_gt += 1
                if any(x1 <= cx <= x2 and y1 <= cy <= y2 for cx, cy in centres):
                    per_class_hit[cls_name] += 1
                    img_hit += 1
            overall_hit += img_hit
            overall_gt += img_gt
            overall_preds += len(centres)
            image_id_int = int(t["image_id"]) if "image_id" in t else (batch_idx * MICRO_BATCH)
            per_image_rows.append({
                "image_idx": image_id_int,
                "gt": img_gt,
                "hit": img_hit,
                "cpv_%": round(100.0 * img_hit / max(1, img_gt), 2),
                "preds": len(centres),
            })

m = metric.compute()
mAP_50    = float(m["map_50"])
mAP_50_95 = float(m["map"])
overall_cpv = 100.0 * overall_hit / max(1, overall_gt)

print(f"REPORT eval | mAP@.5 = {mAP_50:0.4f} | mAP@.5:.95 = {mAP_50_95:0.4f} "
      f"| CPV = {overall_cpv:0.2f}% ({overall_hit}/{overall_gt}) | total_preds = {overall_preds}")

# Per-class print.
print(f"{'class':<12} {'gt':>6} {'hit':>6} {'cpv_%':>8}")
print("-" * 36)
for c in CLASSES:
    gt = per_class_gt[c]
    hit = per_class_hit[c]
    cpv = (100.0 * hit / gt) if gt else 0.0
    print(f"{c:<12} {gt:>6d} {hit:>6d} {cpv:>7.2f}")
print(f"{'OVERALL':<12} {overall_gt:>6d} {overall_hit:>6d} {overall_cpv:>7.2f}")

## 10.5 — Write CSVs for the report

Three CSVs to Drive (so they survive runtime restarts):
- `reports/tables/cpv_summary_detr.csv` — per-class + OVERALL CPV. Same shape as `cpv_summary.csv` from `scripts/run_cpv.py`.
- `reports/tables/cpv_per_image_detr.csv` — per-image hit / total. Same shape as `cpv_per_image.csv`.
- `reports/tables/transfer_experiments.csv` — appended (or written fresh): new row `detr-r50, finetune` so T-01 in the gaps tracker picks up the DETR row.

Copy the CSVs into the cloned repo at `/content/visclick/reports/tables/` as well so the next `git push` from the Windows side carries them.

In [ ]:
import csv, os, shutil

DRIVE_TABLES = os.path.join(DRIVE, "reports", "tables")
REPO_TABLES  = "/content/visclick/reports/tables"
os.makedirs(DRIVE_TABLES, exist_ok=True)
os.makedirs(REPO_TABLES, exist_ok=True)

def _mirror(name: str, write_fn):
    drive_path = os.path.join(DRIVE_TABLES, name)
    write_fn(drive_path)
    shutil.copy2(drive_path, os.path.join(REPO_TABLES, name))
    print("REPORT csv |", name, "->", drive_path, "(mirrored to repo)")

def _write_cpv_summary(path: str):
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["class", "gt", "hit", "cpv_%"])
        for c in CLASSES:
            gt = per_class_gt[c]; hit = per_class_hit[c]
            w.writerow([c, gt, hit, round(100.0 * hit / gt, 2) if gt else 0.0])
        w.writerow(["OVERALL", overall_gt, overall_hit, round(overall_cpv, 2)])

def _write_cpv_per_image(path: str):
    with open(path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=["image_idx", "gt", "hit", "cpv_%", "preds"])
        w.writeheader()
        for row in per_image_rows:
            w.writerow(row)

def _write_transfer(path: str):
    new_row = {
        "model": "detr-r50",
        "stage": "desktop_finetune",
        "n_train": len(hc_ds),
        "n_test":  len(hc_ds),
        "epochs": EPOCHS,
        "imgsz": "DETR-default-800",
        "freeze_backbone": True,
        "lr_head": LR_HEAD,
        "map_50": round(mAP_50, 4),
        "map_50_95": round(mAP_50_95, 4),
        "cpv_overall_%": round(overall_cpv, 2),
        "cpv_conf": CONF_CPV,
        "weights": STABLE,
    }
    existing = []
    if os.path.isfile(path):
        with open(path) as fh:
            reader = csv.DictReader(fh)
            existing = list(reader)
            existing_fields = reader.fieldnames or list(new_row.keys())
    else:
        existing_fields = list(new_row.keys())
    # Ensure new fields are kept.
    fieldnames = list(dict.fromkeys(list(existing_fields) + list(new_row.keys())))
    with open(path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for r in existing:
            w.writerow(r)
        w.writerow(new_row)

_mirror("cpv_summary_detr.csv",        _write_cpv_summary)
_mirror("cpv_per_image_detr.csv",      _write_cpv_per_image)
_mirror("transfer_experiments.csv",    _write_transfer)
print("REPORT step = WRITE_CSV | done")